02 — Nettoyage et préparation des données Fifa
Objectif

Préparer un jeu de données regroupant les résultats des Coupes du monde de football de 1930 à 2022.

Les étapes prévues sont :

    Importer et assembler les fichiers CSV des différentes éditions.

    Vérifier la qualité et la cohérence des données.

    Traiter les éventuelles anomalies.

    Harmoniser les noms des équipes.

    Identifier les pays organisateurs de chaque édition.

Ce jeu de données préparé servira ensuite à analyser les performances des pays hôtes et à étudier leur éventuel avantage sportif.

In [1]:
import pandas as pd
import glob
import os

In [2]:
fichiers_csv = glob.glob("../data/FIFA - *.csv")

print(len(fichiers_csv))

22


In [3]:
liste_df = []

for fichier in fichiers_csv:
    df_temp = pd.read_csv(fichier)

    annee = int(os.path.basename(fichier).replace("FIFA - ", "").replace(".csv", ""))

    df_temp["Year"] = annee

    liste_df.append(df_temp)

In [4]:
df_fifa = pd.concat(liste_df, ignore_index=True)

In [5]:
print(df_fifa.shape)
print(df_fifa["Year"].nunique())

(489, 11)
22


In [6]:
df_fifa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 489 entries, 0 to 488
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Position         489 non-null    int64 
 1   Team             489 non-null    object
 2   Games Played     489 non-null    int64 
 3   Win              489 non-null    int64 
 4   Draw             489 non-null    int64 
 5   Loss             489 non-null    int64 
 6   Goals For        489 non-null    int64 
 7   Goals Against    489 non-null    int64 
 8   Goal Difference  489 non-null    object
 9   Points           489 non-null    int64 
 10  Year             489 non-null    int64 
dtypes: int64(9), object(2)
memory usage: 42.1+ KB


In [7]:
df_fifa["Goal Difference"] = (
    df_fifa["Goal Difference"]
    .astype(str)
    .str.replace("−", "-", regex=False)
    .astype(int)
)

On s'est aperçu lors de la phase d'exploration que pour la colonne Goal difference, les différences de buts négatives étaient indiquées par un tiret non mathématique ce qui passait la colonne en type objet. On remplace donc les tirets par des "moins" mathématiques.

In [8]:
df_fifa["Goal Difference"].dtype

dtype('int64')

On vérifie ensuite que pour chaque équipe, le nombre de matchs joués corresponde bien à la somme des victoires, des matchs nuls et des défaites.

In [9]:
df_fifa["Total Results"] = df_fifa["Win"] + df_fifa["Draw"] + df_fifa["Loss"]

df_fifa[df_fifa["Games Played"] != df_fifa["Total Results"]]

,Position,Team,Games Played,Win,Draw,Loss,Goals For,Goals Against,Goal Difference,Points,Year,Total Results


Non n'avons plus besoin de la colonne Total results, nous pouvons donc la supprimer, Games played suffit. 

In [10]:
df_fifa = df_fifa.drop(columns=["Total Results"])

On vérifie s'il y a des lignes strictement identiques.

In [11]:
df_fifa.duplicated().sum()

np.int64(0)

Il faut maintenant régler un détail important pour notre problématique : harmoniser les noms des pays au fil des 90 ans. Pour cela, on se base sur l'héritage des fédérations officiellement retenu par la FIFA. Par exemple, l'Allemagne depuis 1990 est l'héritière de la RFA, la Russie de l'URSS, la Serbie de la Yougoslavie, ou encore la République Tchèque de la Tchécoslovaquie.

In [12]:
sorted(df_fifa["Team"].unique())

['Algeria',
 'Angola',
 'Argentina',
 'Australia',
 'Austria',
 'Belgium',
 'Bolivia',
 'Bosnia and Herzegovina',
 'Brazil',
 'Bulgaria',
 'Bulgaria**',
 'Cameroon',
 'Canada',
 'Chile',
 'China PR',
 'Colombia',
 'Costa Rica',
 'Croatia',
 'Cuba',
 'Czech Republic',
 'Czechoslovakia',
 'Denmark',
 'Dutch East Indies',
 'East Germany',
 'Ecuador',
 'Egypt',
 'El Salvador',
 'England',
 'FR Yugoslavia',
 'France',
 'Germany',
 'Ghana',
 'Greece',
 'Haiti',
 'Honduras',
 'Hungary',
 'Iceland',
 'Iran',
 'Iraq',
 'Israel*',
 'Italy',
 'Ivory Coast',
 'Jamaica',
 'Japan',
 'Kuwait',
 'Mexico',
 'Morocco',
 'Netherlands',
 'New Zealand',
 'Nigeria',
 'North Korea',
 'Northern Ireland',
 'Norway',
 'Panama',
 'Paraguay',
 'Peru',
 'Poland',
 'Portugal',
 'Qatar',
 'Republic of Ireland',
 'Romania',
 'Russia',
 'Saudi Arabia',
 'Scotland',
 'Senegal',
 'Serbia',
 'Serbia and Montenegro',
 'Slovakia',
 'Slovenia',
 'South Africa',
 'South Korea',
 'Soviet Union',
 'Spain',
 'Sweden',
 'Switzer

In [13]:
for team in sorted(df_fifa["Team"].unique()):
    print(team)

Algeria
Angola
Argentina
Australia
Austria
Belgium
Bolivia
Bosnia and Herzegovina
Brazil
Bulgaria
Bulgaria**
Cameroon
Canada
Chile
China PR
Colombia
Costa Rica
Croatia
Cuba
Czech Republic
Czechoslovakia
Denmark
Dutch East Indies
East Germany
Ecuador
Egypt
El Salvador
England
FR Yugoslavia
France
Germany
Ghana
Greece
Haiti
Honduras
Hungary
Iceland
Iran
Iraq
Israel*
Italy
Ivory Coast
Jamaica
Japan
Kuwait
Mexico
Morocco
Netherlands
New Zealand
Nigeria
North Korea
Northern Ireland
Norway
Panama
Paraguay
Peru
Poland
Portugal
Qatar
Republic of Ireland
Romania
Russia
Saudi Arabia
Scotland
Senegal
Serbia
Serbia and Montenegro
Slovakia
Slovenia
South Africa
South Korea
Soviet Union
Spain
Sweden
Switzerland
Togo
Trinidad and Tobago
Tunisia
Turkey
USA
Ukraine
United Arab Emirates
United States
Uruguay
Wales
West Germany
Yugoslavia
Zaire


In [14]:
df_fifa[df_fifa["Team"].str.contains(r"\*")][["Year", "Team"]]

,Year,Team
132,1970,Israel*
133,1970,Bulgaria**


In [15]:
df_fifa["Team"] = df_fifa["Team"].str.replace("*", "", regex=False).str.strip()

In [16]:
df_fifa[df_fifa["Year"] == 1970][["Year", "Team"]]

,Year,Team
121,1970,Brazil
122,1970,Italy
123,1970,West Germany
124,1970,Uruguay
125,1970,Soviet Union
126,1970,Mexico
127,1970,Peru
128,1970,England
129,1970,Sweden
130,1970,Belgium
